#### working on writing code to generate list of paths for the filenames list in batch kilosort 

In [4]:
#need to import some stuff to make a gui for selecting folders


import tkinter as tk
from tkinter import filedialog
import os



#want to select a series of base folders, have code that generates a list of paths to the continuous.dat files for each of these folders

def find_continuous_dat_files(folder):
    matches = []
    for root_dir, _, files in os.walk(folder):
        for file in files:
            if file == "continuous.dat":
                matches.append(os.path.join(root_dir, file))
    return matches

def main(start_dir=None):
    root = tk.Tk()
    root.withdraw()  # Hide main window

    selected_folders = []
    while True:
        folder = filedialog.askdirectory(
            title="Select a folder (Cancel to finish)",
            initialdir=start_dir if start_dir else os.getcwd()
        )
        if not folder:
            break
        selected_folders.append(folder)
        print(f"Selected folder: {folder}")

    continuous_files = []
    for folder in selected_folders:
        continuous_files.extend(find_continuous_dat_files(folder))

    return continuous_files

if __name__ == "__main__":
    files_list = main(start_dir='/Volumes/Projects/PreyCapture/ZI_A1Activation/') #change this to be the root directory for your data
    print("List of continuous.dat files found:")
    print(files_list)





Selected folder: /Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_10-48-29_mouse-3275
Selected folder: /Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_10-53-46_mouse-3275
Selected folder: /Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_11-00-05_mouse-3275
Selected folder: /Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-06_10-17-03_mouse-3275
List of continuous.dat files found:
['/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_10-48-29_mouse-3275/2025-06-07_10-48-33_mouse-3275/Record Node 102/experiment1/recording1/continuous/OE_FPGA_Acquisition_Board-100.Rhythm Data/continuous.dat', '/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_10-53-46_mouse-3275/2025-06-07_10-53-49_mouse-3275/Record Node 102/experiment1/recording1/continuous/OE_FPGA_Acquisition_Board-100.Rhythm Data/continuous.dat', '/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_11-00-05_mouse-3275/2025-06-07_11-00-09_mouse-3275/Record Node 102/experiment1/recording1/con

In [6]:
#after filelist is generated, run batch kilosort on these folders
from kilosort import run_kilosort
from kilosort.io import BinaryFileGroup, load_probe

# Also add whatever else you need to set for your data, like sampling rate
settings = {'n_chan_bin': 32, 'nblocks' : 0, 'fs' : 30000, 'batch_size' : 60000}
n_channels=32

#set path to probe channel map
probe_path=r'C:\Users\wehrlab\Documents\GitHub\Kilosort3_8TT\chanMap.mat' #change these paths to be what you need
probe = load_probe(probe_path)

#generate a filenames variable that points to your earlier generated files_list
filenames = files_list

data_dir=r'F:\lab\djmaus\data\molly' #change this path to be what you need

# run kilosort on your list of files
run_kilosort(settings=settings, filename=filenames, probe=probe)

['/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_10-48-29_mouse-3275/2025-06-07_10-48-33_mouse-3275/Record Node 102/experiment1/recording1/continuous/OE_FPGA_Acquisition_Board-100.Rhythm Data/continuous.dat',
 '/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_10-53-46_mouse-3275/2025-06-07_10-53-49_mouse-3275/Record Node 102/experiment1/recording1/continuous/OE_FPGA_Acquisition_Board-100.Rhythm Data/continuous.dat',
 '/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_11-00-05_mouse-3275/2025-06-07_11-00-09_mouse-3275/Record Node 102/experiment1/recording1/continuous/OE_FPGA_Acquisition_Board-100.Rhythm Data/continuous.dat',
 '/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-06_10-17-03_mouse-3275/2025-06-06_10-17-08_mouse-3275/Record Node 102/experiment1/recording1/continuous/OE_FPGA_Acquisition_Board-100.Rhythm Data/continuous.dat']

In [16]:
file_path='/Volumes/Projects/PreyCapture/ZI_A1Activation/2025-06-07_10-48-29_mouse-3275/2025-06-07_10-48-33_mouse-3275/Record Node 102/experiment1/recording1/continuous/OE_FPGA_Acquisition_Board-100.Rhythm Data/continuous.dat'
file_size_bytes = os.path.getsize(file_path)
print(file_size_bytes)
sample_length = file_size_bytes // (2 * 43)
print(sample_length)
recording_time=sample_length/30000
print(recording_time)


783483564
9110274
303.6758


In [22]:
(5773480/30000)/60

3.207488888888889

In [23]:
0.207488888888889*60

12.44933333333334

In [9]:
import sys

In [ ]:
import tkinter as tk
from tkinter import filedialog
import os
import json

def find_continuous_dat_files(folder):
    matches = []
    for root_dir, _, files in os.walk(folder):
        for file in files:
            if file == "continuous.dat":
                matches.append(os.path.join(root_dir, file))
    return matches

def select_folders(start_dir=None):
    root = tk.Tk()
    root.withdraw()  # Hide main window

    selected_folders = []
    while True:
        folder = filedialog.askdirectory(
            title="Select a folder (Cancel to finish)",
            initialdir=start_dir if start_dir else os.getcwd()
        )
        if not folder:
            break
        selected_folders.append(folder)
        print(f"Selected folder: {folder}")
        start_dir = folder
    return selected_folders

def get_file_sample_length(file_path, n_chan_bin):
    try:
        file_size_bytes = os.path.getsize(file_path)
        sample_length = file_size_bytes // (2 * n_chan_bin)  # 2 bytes per int16 sample
        return sample_length
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def main(start_dir, data_dir, probe_path, n_chan_bin):
    selected_folders = select_folders(start_dir)

    metadata = {}

    for folder in selected_folders:
        continuous_files = find_continuous_dat_files(folder)
        for file_path in continuous_files:
            length_samples = get_file_sample_length(file_path, n_chan_bin)
            metadata[file_path] = {
                "length_samples": length_samples,
                "folder": folder
            }

    # Save metadata to JSON
    if selected_folders:
        output_json = os.path.join(selected_folders[0], "recording_lengths.json")
        with open(output_json, 'w') as f:
            json.dump(metadata, f, indent=4)
        print(f"\nSaved metadata to: {output_json}")
    else:
        print("No folders selected, metadata not saved.")
    # Example: Pass file list to Kilosort if needed
    from kilosort import run_kilosort
    from kilosort.io import BinaryFileGroup, load_probe

    settings = {'n_chan_bin': n_chan_bin, 'nblocks': 0, 'fs': 30000, 'batch_size': 60000}
    probe = load_probe(probe_path)

    file_list = list(metadata.keys())

    print("\nStarting kilosort batch...")
    run_kilosort(settings=settings, filename=file_list, probe=probe)
    print("Kilosort batch finished.")

    return metadata

if __name__ == "__main__":
    start_dir =  "\\wehr-nas.uoregon.edu\\Projects\\PreyCapture\\ZI_A1Activation"
    data_dir =   "\\wehr-nas.uoregon.edu\\Projects\\PreyCapture\\ZI_A1Activation"
    probe_path = "C:\\Users\\wehrlab\\Documents\\GitHub\\Kilosort3_8TT\\chanMap.mat"
    n_chan_bin = 43

    metadata = main(start_dir=start_dir, data_dir=data_dir, probe_path=probe_path, n_chan_bin=n_chan_bin)


TypeError: main() missing 1 required positional argument: 'probe_path'